In [1]:
# !pip install transformers
# !pip install "transformers[torch]"

In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

C:\ProgramData\anaconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data = pd.read_csv("samsum-train.csv")
validation_data = pd.read_csv("samsum-validation.csv")

train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [4]:
print(f"train shape : ", train_data.shape)
print(f"validation shape : ", validation_data.shape)
train_data["dialogue"][0]


train shape :  (14732, 3)
validation shape :  (818, 3)


"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

# Random Sampling

In [5]:
# random sampling
train_data = train_data.sample(n=14732, random_state=42).reset_index(drop=True)
val_data = validation_data.sample(n=818, random_state=42).reset_index(drop=True)
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting\r\nViolet: <file_other>\r\nClaire: Hi! :) Thanks, but I've already read it. :)\r\nClaire: But thanks for thinking about me :)"

# Data Pre Processing

In [6]:
# Removing new line and spaces and html tags
# . : means replace any simgle charcater, * : means repeat previous pattern any number of times
# .* : means replace any character any number of times (greedy match) tryies to match as much as possible.
# ? : means lazy match non greedy match as less as possible.

import re

def clean_data(text):
    # \r : moves cursor back to start line
    re.sub(r"\r\n", " ", text) # lines replaced by single space
    re.sub(r"\s+", " ", text)  # multi space replaced by single space
    re.sub(r"<.*?>", " ", text)  # replacing html tages (non greedy)
    text.strip().lower()  # removes trainling spaces and lowercase to all text
    return text
    
    

In [7]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [8]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting\r\nViolet: <file_other>\r\nClaire: Hi! :) Thanks, but I've already read it. :)\r\nClaire: But thanks for thinking about me :)"

In [9]:
# tokenizing the input to convert the raw data into numeric forms for fine tuning
# tokenizer need 4 prameters: data, padding, max_length, truncate
tokenizer = T5Tokenizer.from_pretrained("t5-small") 

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)  # returns a object with input ids and attention_mask
    # print(f"inputs => {inputs}")
    target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)
    # print(f"token check target id => {target}")
    
    inputs["labels"] = target["input_ids"]
    return inputs
    

In [10]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [11]:
print(type(train_dataset))
train_dataset
# input_ids :  dialogue token ids
# labels :  summary token ids

<class 'list'>


[{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 3, 2, 11966, 834, 9269, 3155, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# fine-tune model

In [12]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 7703.09it/s]


In [13]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
else:
    device = torch.device("cpu") 

print(f"Dvice : {device}")
model.to(device)

Dvice : cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500  # for 500 steps our lr will increase from 0 to the real defined lr value
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 